# RSNA Knee Abnormality Detection — EDA

First look at `train.csv` and `train_series.csv`: label prevalence, how
many studies have human-annotated labels vs. report-only, series-per-study
and anatomical-plane/sequence-type distribution, patient sex, report
language, and slice counts per series. Runs on Kaggle only — the dataset
(569.76 GB) is never downloaded locally; see `docs/1_instructions.md`.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "v1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
    SRC_DATASET_DIR = Path("/kaggle/input/rsna-knee-mri-src")
    if SRC_DATASET_DIR.exists():
        sys.path.insert(0, str(SRC_DATASET_DIR / "src"))
else:
    DATA_DIR = Path("../data")
    sys.path.insert(0, str(Path("../src").resolve()))

print(f"NOTEBOOK_VERSION={NOTEBOOK_VERSION}")
print(f"IS_KAGGLE={IS_KAGGLE}")
print(f"DATA_DIR={DATA_DIR}")

## 1. Load study & series tables

In [ ]:
from knee_mri.labels import LABEL_COLUMNS
from knee_mri.dataset import split_labeled_studies

train_df = pd.read_csv(DATA_DIR / "train.csv")
series_df = pd.read_csv(DATA_DIR / "train_series.csv")

print(train_df.shape, series_df.shape)
train_df.head()

## 2. Label prevalence and labeled-vs-unlabeled split

In [ ]:
import matplotlib.pyplot as plt

labeled, unlabeled = split_labeled_studies(train_df)
print(f"Labeled studies: {len(labeled)}")
print(f"Unlabeled (report-only) studies: {len(unlabeled)}")

prevalence = labeled[LABEL_COLUMNS].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
prevalence.plot(kind="barh", ax=ax, colormap="viridis")
ax.set_xlabel("Positive rate (labeled studies)")
ax.set_title("Label prevalence across the 12 targets")
plt.tight_layout()
plt.show()

prevalence

**Insight:** pending first Kaggle run.

## 3. Series per study, anatomical plane, and sequence type

In [ ]:
series_per_study = series_df.groupby("StudyInstanceUID").size()

fig, ax = plt.subplots(figsize=(8, 4))
series_per_study.plot(kind="hist", bins=30, ax=ax, colormap="viridis")
ax.set_xlabel("Series per study")
ax.set_title("Distribution of series count per study")
plt.tight_layout()
plt.show()

print(series_df["Anatomical_Plane"].value_counts())
print(series_df[["Fluid_Sensitive", "Fat_Suppression"]].mean())

**Insight:** pending first Kaggle run.

## 4. Patient sex and report language spot-check

In [ ]:
print(train_df["PatientSex"].value_counts(dropna=False))

sample_reports = train_df["Report"].dropna().sample(5, random_state=SEED)
for idx, report in sample_reports.items():
    non_ascii_fraction = sum(ord(ch) > 127 for ch in report) / max(len(report), 1)
    study_id = train_df.loc[idx, "StudyInstanceUID"]
    print(f"--- {study_id} (len={len(report)}, non_ascii_fraction={non_ascii_fraction:.2f}) ---")
    print(report[:300])
    print()

**Insight:** pending first Kaggle run.

## 5. Slice count per series

In [ ]:
train_series_root = DATA_DIR / "train_series"

slice_counts = []
for study_dir in sorted(train_series_root.iterdir())[:200]:  # cap for a fast EDA pass
    for series_dir in study_dir.iterdir():
        slice_counts.append(len(list(series_dir.glob("*.dcm"))))

slice_counts = pd.Series(slice_counts)
print(slice_counts.describe())

fig, ax = plt.subplots(figsize=(8, 4))
slice_counts.plot(kind="hist", bins=30, ax=ax, colormap="viridis")
ax.set_xlabel("Slices per series")
ax.set_title("Distribution of slice count per series (first 200 studies)")
plt.tight_layout()
plt.show()

**Insight:** pending first Kaggle run.